In [17]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

from google import genai
from google.genai.types import (
    LiveConnectConfig,
    SpeechConfig,
    VoiceConfig,
    PrebuiltVoiceConfig
)

from google.genai.types import Content, Part

import soundfile as sf
import numpy as np

import nest_asyncio



load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)


model = "gemini-2.0-flash-live-001"

In [18]:
voice_name = "Aoede"


audio_config = LiveConnectConfig(
    response_modalities=["AUDIO"],
    speech_config=SpeechConfig(
        voice_config=VoiceConfig(
            prebuilt_voice_config=PrebuiltVoiceConfig(voice_name=voice_name)
        )
    ),
    system_instruction="you are a super friendly, sometime a bit to friendly assistant",
)

# Managing Audio I/O with our **AudioManager**

- To keep our audio input and output logic clean and organized, I create an AudioManager class. This class will be responsible for:

- Initializing PyAudio for microphone input and speaker output.
- Buffering and playing audio chunks received from Gemini.
- Handling interruptions.

In [19]:
import pyaudio
from collections import deque

FORMAT = pyaudio.paInt16  # Audio format: 16-bit PCM
SEND_SAMPLE_RATE = 16000  # Sample rate for audio sent to Gemini (Hz)
RECEIVE_SAMPLE_RATE = 24000 # Sample rate for audio received from Gemini (Hz)
CHUNK_SIZE = 512          # Size of audio chunks to process
CHANNELS = 1              # Mono audio

In [20]:
class AudioManager:
    def __init__(self, input_sample_rate=16000, output_sample_rate=24000):
        self.pya = pyaudio.PyAudio()
        self.input_stream = None
        self.output_stream = None
        self.input_sample_rate = input_sample_rate
        self.output_sample_rate = output_sample_rate
        self.audio_queue = deque()
        self.is_playing = False
        self.playback_task = None
        
    async def initialize(self):
        mic_info = self.pya.get_default_input_device_info()
        print(f"microphone used: {mic_info}")

        self.input_stream = await asyncio.to_thread(
            self.pya.open,
            format=FORMAT,
            channels=CHANNELS,
            rate=self.input_sample_rate,
            input=True,
            input_device_index=mic_info["index"],
            frames_per_buffer=CHUNK_SIZE,
        )

        self.output_stream = await asyncio.to_thread(
            self.pya.open,
            format=FORMAT,
            channels=CHANNELS,
            rate=self.output_sample_rate,
            output=True,
        )
        
    def add_audio(self, audio_data):
        """Adds received audio data to the playback queue."""
        self.audio_queue.append(audio_data)
        # If playback isn't running, start it
        if self.playback_task is None or self.playback_task.done():
            self.playback_task = asyncio.create_task(self._play_audio())
            
    async def _play_audio(self):
        """Plays audio chunks from the queue."""
        print("🗣️ Gemini talking...")
        while self.audio_queue:
            try:
                audio_data = self.audio_queue.popleft()
                await asyncio.to_thread(self.output_stream.write, audio_data)
            except Exception as e:
                print(f"Error playing audio: {e}")
                break # Stop playback on error
        print("Playback queue empty.")
        self.playback_task = None # Reset task when done
        
    def interrupt(self):
        """Handle interruption by stopping playback and clearing queue"""
        self.audio_queue.clear()
        self.is_playing = False

        # Important: Start a clean state for next response
        if self.playback_task and not self.playback_task.done():
            self.playback_task.cancel()
            
            

# The main **audio_loop**

- listen_for_audio(): Captures audio from your microphone.
- process_and_send_audio(): Sends your captured audio to Gemini.
- receive_and_play(): Receives Gemini's audio response and plays it.

In [21]:
import asyncio

In [22]:
async def audio_loop():
    audio_manager = AudioManager(
        input_sample_rate=SEND_SAMPLE_RATE, output_sample_rate=RECEIVE_SAMPLE_RATE
    )

    await audio_manager.initialize()

    async with (
        client.aio.live.connect(model=model, config=audio_config) as session,
        asyncio.TaskGroup() as tg,
    ):
    
        audio_queue = asyncio.Queue()

        async def listen_for_audio():
            """Just captures audio and puts it in the queue"""
            while True:
                    data = await asyncio.to_thread(
                        audio_manager.input_stream.read,
                        CHUNK_SIZE,
                        exception_on_overflow=False,
                    )
                    await audio_queue.put(data)
                    
        async def process_and_send_audio():
            """Processes audio from queue and sends to Gemini"""
            while True:
                data = await audio_queue.get()

                # Always send the audio data to Gemini
                await session.send_realtime_input(
                    media={
                        "data": data,
                        "mime_type": f"audio/pcm;rate={SEND_SAMPLE_RATE}",
                    }
                )

                audio_queue.task_done()
                            
        async def receive_and_play():
                while True:

                    async for response in session.receive():
                        server_content = response.server_content

                        if (
                            hasattr(server_content, "interrupted")
                            and server_content.interrupted
                        ):
                            print(f"🤐 INTERRUPTION DETECTED")
                            audio_manager.interrupt()

                        if server_content and server_content.model_turn:
                            for part in server_content.model_turn.parts:
                                if part.inline_data:
                                    audio_manager.add_audio(part.inline_data.data)

                        if server_content and server_content.turn_complete:
                            print("✅ Gemini done talking")
                            
             # Start all tasks with proper task creation
        tg.create_task(listen_for_audio())
        tg.create_task(process_and_send_audio())
        tg.create_task(receive_and_play())
        
    
    async for response in session.receive():
        if response.server_content and response.server_content.interrupted is True:
            # The generation was interrupted by the user.
            audio_manager.interrupt()

In [ ]:
nest_asyncio.apply()
await audio_loop()